In [ ]:
# colabならこれだけ入れれば動くはず
!pip install minari
!pip install stable-baselines3[extra]
!pip install sb3-contrib

In [ ]:
import numpy as np
import pickle
import tqdm
import torch
from stable_baselines3 import PPO, SAC, TD3, A2C
from sb3_contrib import TQC, TRPO, ARS
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from utils import log_likelihoods, double_centering

In [ ]:
# 対数尤度計算用データセットの読み込み(minari)
import minari


states = []
actions = []
rewards = []

dataset1 = minari.load_dataset("mujoco/halfcheetah/simple-v0", download=True)
dataset2 = minari.load_dataset("mujoco/halfcheetah/medium-v0", download=True)
dataset3 = minari.load_dataset("mujoco/halfcheetah/expert-v0", download=True)

simples = len(dataset1)
mediums = len(dataset2)
experts = len(dataset3)


for data in dataset1:
    states.append(data.observations[:-1])
    actions.append(data.actions)
    rewards.append(data.rewards)
for data in dataset2:
    states.append(data.observations[:-1])
    actions.append(data.actions)
    rewards.append(data.rewards)
for data in dataset3:
    states.append(data.observations[:-1])
    actions.append(data.actions)
    rewards.append(data.rewards)

In [ ]:
# モデルの特徴量計算

def learning_model_features(model_name, model_path, alg, num, interval=10000):
    features = {}
    for i in tqdm.tqdm(range(num)):
        step = (1+i)*interval
        model = alg.load(model_path+"halfcheetah_model_"+str(step)+"_steps")
        feat = [0 for _ in range(len(states))]
        for t in range(len(states)):
            with torch.no_grad():
                pred_actions, _ = model.predict(states[t])
                real_actions = actions[t]
                errors = real_actions - pred_actions
                llh = log_likelihoods(errors)
                feat[t] = np.mean(llh)
        features[model_name+"_"+str(step)] = feat
    return features

models = {
    "ppo": ("samplemodel/ppo/", PPO, 10, 1000000),
    "td3": ("samplemodel/td3/", TD3, 10, 1000000),
    "a2c": ("samplemodel/a2c/", A2C, 10, 1000000),
    "trpo": ("samplemodel/trpo/", TRPO, 10, 1000000),
    "ars": ("samplemodel/ars/", ARS, 10, 1000000),
    "sac": ("samplemodel/sac/", SAC, 10, 1000000),
    "tqc": ("samplemodel/tqc/", TQC, 10, 1000000),
    
}

all_features = {}

for model_name in models.keys():
    path, alg, num, interval = models[model_name]
    all_features.update(learning_model_features(model_name, path, alg, num, interval))

all_names = list(all_features.keys())
centered = double_centering(np.array([all_features[name] for name in all_names]))

In [ ]:
# マッピング
def mapping_features(centered_features, model_names, color_option="model_type"):
    feature_matrix = StandardScaler().fit_transform(centered_features)
    # t-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=10,
        learning_rate="auto",
        init="pca",
        random_state=0,
        max_iter=1000,
        verbose=1
    )
    mapped_features = tsne.fit_transform(feature_matrix)
    colors = []
    handles=[]
    if color_option == "model_type":
        custom_colors = {
            "ppo": "red",
            "sac": "purple",
            "td3": "green",
            "tqc": "blue",
            "trpo": "yellow",
            "a2c": "orange",
            "ars": "gray",
        }
        for name in model_names:
            for key in custom_colors:
                if key in name.lower():
                    colors.append(custom_colors[key])
                    break
            else:
                colors.append("gray")  # 未分類はグレーに
        handles = [
            plt.Line2D([0], [0], marker='o', color='w', label=label,
                        markerfacecolor=color, markersize=8)
            for label, color in custom_colors.items()
        ]
    
    else:
        for name in model_names:
            colors.append(0)

    # 結果をプロット
    plt.figure(figsize=(8, 6))
    x_coords = mapped_features[:, 0]
    y_coords = mapped_features[:, 1]

    for i in range(len(colors)):
        plt.scatter(x_coords[i], y_coords[i], color=colors[i], s=18)
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.grid(True)
    plt.legend(handles=handles)
    plt.tight_layout()
    plt.show()

In [ ]:
mapping_features(centered, all_names)